In [2]:
import pandas as pd
df = pd.read_csv("mental_health_data final data.csv")
print(df.shape)
print(df.columns)

(50000, 17)
Index(['User_ID', 'Age', 'Gender', 'Occupation', 'Country',
       'Mental_Health_Condition', 'Severity', 'Consultation_History',
       'Stress_Level', 'Sleep_Hours', 'Work_Hours', 'Physical_Activity_Hours',
       'Social_Media_Usage', 'Diet_Quality', 'Smoking_Habit',
       'Alcohol_Consumption', 'Medication_Usage'],
      dtype='object')


In [3]:
df.dtypes

User_ID                      int64
Age                          int64
Gender                      object
Occupation                  object
Country                     object
Mental_Health_Condition     object
Severity                    object
Consultation_History        object
Stress_Level                object
Sleep_Hours                float64
Work_Hours                   int64
Physical_Activity_Hours      int64
Social_Media_Usage         float64
Diet_Quality                object
Smoking_Habit               object
Alcohol_Consumption         object
Medication_Usage            object
dtype: object

In [4]:
df.isnull().sum()

User_ID                        0
Age                            0
Gender                         0
Occupation                     0
Country                        0
Mental_Health_Condition        0
Severity                   25002
Consultation_History           0
Stress_Level                   0
Sleep_Hours                    0
Work_Hours                     0
Physical_Activity_Hours        0
Social_Media_Usage             0
Diet_Quality                   0
Smoking_Habit                  0
Alcohol_Consumption            0
Medication_Usage               0
dtype: int64

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

# Define target and features
y = df['Stress_Level']

# Select features (you can customize this list)
features = ['Age', 'Gender', 'Occupation', 'Country',
            'Sleep_Hours', 'Work_Hours', 'Physical_Activity_Hours',
            'Social_Media_Usage', 'Diet_Quality', 'Smoking_Habit',
            'Alcohol_Consumption', 'Medication_Usage']

X = df[features]

# One-hot encode categorical features
categorical_cols = X.select_dtypes(include='object').columns.tolist()
X_encoded = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

# Scale numeric features
numeric_cols = ['Age', 'Sleep_Hours', 'Work_Hours', 'Physical_Activity_Hours', 'Social_Media_Usage']
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train[numeric_cols])
X_test_scaled = scaler.transform(X_test[numeric_cols])

# Convert scaled arrays back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled, columns=numeric_cols, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=numeric_cols, index=X_test.index)

# Replace original columns with scaled ones
X_train_final = X_train.copy()
X_test_final = X_test.copy()

X_train_final[numeric_cols] = X_train_scaled
X_test_final[numeric_cols] = X_test_scaled

# Train RandomForest model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_final, y_train)

# Predict class labels
y_pred = model.predict(X_test_final)

# Predict class probabilities
y_proba = model.predict_proba(X_test_final)

# Evaluation
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

# Compute multiclass AUC
auc = roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro')
print(f"AUC Score (multiclass, macro-average): {auc:.4f}")


Classification Report:

              precision    recall  f1-score   support

        High       0.34      0.34      0.34      3406
         Low       0.32      0.32      0.32      3280
      Medium       0.33      0.33      0.33      3314

    accuracy                           0.33     10000
   macro avg       0.33      0.33      0.33     10000
weighted avg       0.33      0.33      0.33     10000

AUC Score (multiclass, macro-average): 0.4996
